# 01. Feature Engineering

Notebook này thực hiện xây dựng các đặc trưng phục vụ mô hình dự báo CPI nhóm Giao thông.

Các đặc trưng được xây dựng dựa trên kết quả EDA, bao gồm:

- Độ trễ của CPI.
- Biến động theo tháng của giá nhiên liệu, dầu thô và tỷ giá.
- Đặc trưng trung bình trượt ngắn hạn.
- Đặc trưng mùa vụ.
- Các biến giả Tết và Covid-19.
- Một số đặc trưng phản ánh quan hệ giữa các biến giá dầu.

Các đặc trưng sử dụng cho dự báo được xây dựng từ thông tin quá khứ nhằm hạn chế rò rỉ dữ liệu tương lai.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/model_dataset.csv"
)

df["MonthYear"] = pd.to_datetime(
    df["MonthYear"]
).dt.to_period("M")

df.head()

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid
0,2011-09,NaN,21800.0,21100.0,112.83,85.52,20628.0,0,0
1,2011-10,NaN,21800.0,20400.0,109.55,86.32,20713.0,0,0
2,2011-11,-0.01,21800.0,20400.0,110.77,97.16,20803.0,0,0
3,2011-12,0.16,21800.0,20400.0,107.87,98.56,20813.6,0,0
4,2012-01,0.66,21800.0,20400.0,110.69,100.27,20828.0,1,0


In [4]:
change_cols = [
    "RON95",
    "Diesel",
    "Brent",
    "WTI",
    "USD_VND"
]

for col in change_cols:
    df[f"{col}_change"] = (
        df[col]
        .pct_change(fill_method=None)
        .mul(100)
    )

df[
    [
        "MonthYear",
        "CPI",
        "RON95_change",
        "Diesel_change",
        "Brent_change",
        "WTI_change",
        "USD_VND_change"
    ]
].head()

,MonthYear,CPI,RON95_change,Diesel_change,Brent_change,WTI_change,USD_VND_change
0,2011-09,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-10,NaN,0.0,-3.317536,-2.907028,0.935454,0.412061
2,2011-11,-0.01,0.0,0.000000,1.113647,12.557924,0.434510
3,2011-12,0.16,0.0,0.000000,-2.618037,1.440922,0.050954
4,2012-01,0.66,0.0,0.000000,2.614258,1.734984,0.069186


## 2. Xây dựng các đặc trưng độ trễ

Các biến độ trễ được tạo nhằm cung cấp cho mô hình thông tin của những tháng trước để dự báo CPI tháng hiện tại.

Trong đó:

- `CPI_lag1`: CPI của tháng trước.
- `CPI_lag2`: CPI của 2 tháng trước.
- Các biến `change_lag1`: mức thay đổi của biến kinh tế ở tháng trước.

Việc sử dụng các giá trị quá khứ giúp hạn chế sử dụng thông tin của chính tháng cần dự báo.

In [6]:
# Sắp xếp dữ liệu theo thời gian
df = df.sort_values("MonthYear").reset_index(drop=True)

# Lag của CPI
df["CPI_lag1"] = df["CPI"].shift(1)
df["CPI_lag2"] = df["CPI"].shift(2)

# Lag 1 của các biến kinh tế
df["RON95_change_lag1"] = df["RON95_change"].shift(1)
df["Diesel_change_lag1"] = df["Diesel_change"].shift(1)
df["Brent_change_lag1"] = df["Brent_change"].shift(1)
df["USD_VND_change_lag1"] = df["USD_VND_change"].shift(1)

df.head(10)

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid,RON95_change,Diesel_change,Brent_change,WTI_change,USD_VND_change,CPI_lag1,CPI_lag2,RON95_change_lag1,Diesel_change_lag1,Brent_change_lag1,USD_VND_change_lag1
0,2011-09,NaN,21800.00,21100.00,112.83,85.52,20628.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-10,NaN,21800.00,20400.00,109.55,86.32,20713.0,0,0,0.000000,-3.317536,-2.907028,0.935454,0.412061,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-11,-0.01,21800.00,20400.00,110.77,97.16,20803.0,0,0,0.000000,0.000000,1.113647,12.557924,0.434510,NaN,NaN,0.000000,-3.317536,-2.907028,0.412061
3,2011-12,0.16,21800.00,20400.00,107.87,98.56,20813.6,0,0,0.000000,0.000000,-2.618037,1.440922,0.050954,-0.01,NaN,0.000000,0.000000,1.113647,0.434510
4,2012-01,0.66,21800.00,20400.00,110.69,100.27,20828.0,1,0,0.000000,0.000000,2.614258,1.734984,0.069186,0.16,-0.01,0.000000,0.000000,-2.618037,0.050954
5,2012-02,0.23,21800.00,20400.00,119.33,102.20,20828.0,0,0,0.000000,0.000000,7.805583,1.924803,0.000000,0.66,0.16,0.000000,0.000000,2.614258,0.069186
6,2012-03,1.08,23090.32,21206.45,125.45,106.16,20828.0,0,0,5.918899,3.953186,5.128635,3.874755,0.000000,0.23,0.66,0.000000,0.000000,7.805583,0.000000
7,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0,1.341168,0.912694,-4.543643,-2.675207,0.000000,1.08,0.23,5.918899,3.953186,5.128635,0.000000
8,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0,0.523846,0.150748,-7.858038,-8.381727,0.000000,2.67,1.08,1.341168,0.912694,-4.543643,0.000000
9,2012-06,-1.64,22326.67,20506.67,95.16,82.30,20828.0,0,0,-5.084094,-4.318677,-13.757477,-13.057258,0.000000,1.32,2.67,0.523846,0.150748,-7.858038,0.000000


## 3. Xây dựng đặc trưng trung bình trượt 3 tháng

Các đặc trưng MA3 được tạo nhằm phản ánh xu hướng biến động ngắn hạn của giá nhiên liệu, dầu thô và tỷ giá.

Để tránh sử dụng thông tin của tháng cần dự báo, MA3 tại tháng hiện tại được tính từ **3 tháng trước đó** (`t-1`, `t-2`, `t-3`).

In [7]:
df["RON95_MA3"] = (
    df["RON95_change"]
    .shift(1)
    .rolling(3)
    .mean()
)

df["Diesel_MA3"] = (
    df["Diesel_change"]
    .shift(1)
    .rolling(3)
    .mean()
)

df["Brent_MA3"] = (
    df["Brent_change"]
    .shift(1)
    .rolling(3)
    .mean()
)

df["USDVND_MA3"] = (
    df["USD_VND_change"]
    .shift(1)
    .rolling(3)
    .mean()
)

df.head(10)

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid,RON95_change,...,CPI_lag1,CPI_lag2,RON95_change_lag1,Diesel_change_lag1,Brent_change_lag1,USD_VND_change_lag1,RON95_MA3,Diesel_MA3,Brent_MA3,USDVND_MA3
0,2011-09,NaN,21800.00,21100.00,112.83,85.52,20628.0,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-10,NaN,21800.00,20400.00,109.55,86.32,20713.0,0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-11,-0.01,21800.00,20400.00,110.77,97.16,20803.0,0,0,0.000000,...,NaN,NaN,0.000000,-3.317536,-2.907028,0.412061,NaN,NaN,NaN,NaN
3,2011-12,0.16,21800.00,20400.00,107.87,98.56,20813.6,0,0,0.000000,...,-0.01,NaN,0.000000,0.000000,1.113647,0.434510,NaN,NaN,NaN,NaN
4,2012-01,0.66,21800.00,20400.00,110.69,100.27,20828.0,1,0,0.000000,...,0.16,-0.01,0.000000,0.000000,-2.618037,0.050954,0.000000,-1.105845,-1.470473,0.299175
5,2012-02,0.23,21800.00,20400.00,119.33,102.20,20828.0,0,0,0.000000,...,0.66,0.16,0.000000,0.000000,2.614258,0.069186,0.000000,0.000000,0.369956,0.184883
6,2012-03,1.08,23090.32,21206.45,125.45,106.16,20828.0,0,0,5.918899,...,0.23,0.66,0.000000,0.000000,7.805583,0.000000,0.000000,0.000000,2.600601,0.040047
7,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0,1.341168,...,1.08,0.23,5.918899,3.953186,5.128635,0.000000,1.972966,1.317729,5.182825,0.023062
8,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0,0.523846,...,2.67,1.08,1.341168,0.912694,-4.543643,0.000000,2.420022,1.621960,2.796858,0.000000
9,2012-06,-1.64,22326.67,20506.67,95.16,82.30,20828.0,0,0,-5.084094,...,1.32,2.67,0.523846,0.150748,-7.858038,0.000000,2.594638,1.672209,-2.424349,0.000000


## 4. Xây dựng đặc trưng mùa vụ theo tháng

Do dữ liệu có tần suất theo tháng, hai biến `Month_sin` và `Month_cos` được sử dụng để biểu diễn chu kỳ 12 tháng.

Cách biểu diễn này giúp mô hình nhận biết tính tuần hoàn của thời gian, trong đó tháng 12 và tháng 1 được xem là hai thời điểm liền kề trong chu kỳ năm.

In [8]:
# Lấy số tháng
df["Month"] = df["MonthYear"].dt.month

# Biểu diễn chu kỳ 12 tháng
df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)
df.head(10)

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid,RON95_change,...,Diesel_change_lag1,Brent_change_lag1,USD_VND_change_lag1,RON95_MA3,Diesel_MA3,Brent_MA3,USDVND_MA3,Month,Month_sin,Month_cos
0,2011-09,NaN,21800.00,21100.00,112.83,85.52,20628.0,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,-1.000000e+00,-1.836970e-16
1,2011-10,NaN,21800.00,20400.00,109.55,86.32,20713.0,0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,-8.660254e-01,5.000000e-01
2,2011-11,-0.01,21800.00,20400.00,110.77,97.16,20803.0,0,0,0.000000,...,-3.317536,-2.907028,0.412061,NaN,NaN,NaN,NaN,11,-5.000000e-01,8.660254e-01
3,2011-12,0.16,21800.00,20400.00,107.87,98.56,20813.6,0,0,0.000000,...,0.000000,1.113647,0.434510,NaN,NaN,NaN,NaN,12,-2.449294e-16,1.000000e+00
4,2012-01,0.66,21800.00,20400.00,110.69,100.27,20828.0,1,0,0.000000,...,0.000000,-2.618037,0.050954,0.000000,-1.105845,-1.470473,0.299175,1,5.000000e-01,8.660254e-01
5,2012-02,0.23,21800.00,20400.00,119.33,102.20,20828.0,0,0,0.000000,...,0.000000,2.614258,0.069186,0.000000,0.000000,0.369956,0.184883,2,8.660254e-01,5.000000e-01
6,2012-03,1.08,23090.32,21206.45,125.45,106.16,20828.0,0,0,5.918899,...,0.000000,7.805583,0.000000,0.000000,0.000000,2.600601,0.040047,3,1.000000e+00,6.123234e-17
7,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0,1.341168,...,3.953186,5.128635,0.000000,1.972966,1.317729,5.182825,0.023062,4,8.660254e-01,-5.000000e-01
8,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0,0.523846,...,0.912694,-4.543643,0.000000,2.420022,1.621960,2.796858,0.000000,5,5.000000e-01,-8.660254e-01
9,2012-06,-1.64,22326.67,20506.67,95.16,82.30,20828.0,0,0,-5.084094,...,0.150748,-7.858038,0.000000,2.594638,1.672209,-2.424349,0.000000,6,1.224647e-16,-1.000000e+00


## 5. Xây dựng đặc trưng chênh lệch Brent - WTI

`Brent_WTI_Spread` phản ánh mức chênh lệch giữa giá dầu Brent và WTI.

Do Brent và WTI có tương quan rất cao, việc sử dụng chênh lệch giữa hai loại dầu giúp bổ sung thông tin về trạng thái tương đối của thị trường dầu mà không cần đưa đồng thời cả hai biến ở dạng tương tự vào mô hình.

Để tránh sử dụng thông tin của tháng cần dự báo, đặc trưng được sử dụng ở độ trễ 1 tháng.

In [10]:
df["Brent_WTI_Spread"] = (
    df["Brent"] - df["WTI"]
)

df["Brent_WTI_Spread_lag1"] = (
    df["Brent_WTI_Spread"]
    .shift(1)
)

df.head(10)

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid,RON95_change,...,USD_VND_change_lag1,RON95_MA3,Diesel_MA3,Brent_MA3,USDVND_MA3,Month,Month_sin,Month_cos,Brent_WTI_Spread,Brent_WTI_Spread_lag1
0,2011-09,NaN,21800.00,21100.00,112.83,85.52,20628.0,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,9,-1.000000e+00,-1.836970e-16,27.31,NaN
1,2011-10,NaN,21800.00,20400.00,109.55,86.32,20713.0,0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,10,-8.660254e-01,5.000000e-01,23.23,27.31
2,2011-11,-0.01,21800.00,20400.00,110.77,97.16,20803.0,0,0,0.000000,...,0.412061,NaN,NaN,NaN,NaN,11,-5.000000e-01,8.660254e-01,13.61,23.23
3,2011-12,0.16,21800.00,20400.00,107.87,98.56,20813.6,0,0,0.000000,...,0.434510,NaN,NaN,NaN,NaN,12,-2.449294e-16,1.000000e+00,9.31,13.61
4,2012-01,0.66,21800.00,20400.00,110.69,100.27,20828.0,1,0,0.000000,...,0.050954,0.000000,-1.105845,-1.470473,0.299175,1,5.000000e-01,8.660254e-01,10.42,9.31
5,2012-02,0.23,21800.00,20400.00,119.33,102.20,20828.0,0,0,0.000000,...,0.069186,0.000000,0.000000,0.369956,0.184883,2,8.660254e-01,5.000000e-01,17.13,10.42
6,2012-03,1.08,23090.32,21206.45,125.45,106.16,20828.0,0,0,5.918899,...,0.000000,0.000000,0.000000,2.600601,0.040047,3,1.000000e+00,6.123234e-17,19.29,17.13
7,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0,1.341168,...,0.000000,1.972966,1.317729,5.182825,0.023062,4,8.660254e-01,-5.000000e-01,16.43,19.29
8,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0,0.523846,...,0.000000,2.420022,1.621960,2.796858,0.000000,5,5.000000e-01,-8.660254e-01,15.68,16.43
9,2012-06,-1.64,22326.67,20506.67,95.16,82.30,20828.0,0,0,-5.084094,...,0.000000,2.594638,1.672209,-2.424349,0.000000,6,1.224647e-16,-1.000000e+00,12.86,15.68


## 6. Hoàn thiện bộ đặc trưng

Sau khi xây dựng các đặc trưng, dữ liệu được giới hạn lại trong giai đoạn nghiên cứu từ **01/2012 đến 12/2024**.

Các tháng cuối năm 2011 chỉ được sử dụng làm dữ liệu lịch sử để tính các đặc trưng độ trễ và trung bình trượt, không được đưa vào giai đoạn huấn luyện mô hình.

In [ ]:
feature_cols = [
    # CPI quá khứ
    "CPI_lag1",
    "CPI_lag2",

    # Biến động kinh tế tháng trước
    "RON95_change_lag1",
    "Diesel_change_lag1",
    "Brent_change_lag1",
    "USD_VND_change_lag1",

    # Xu hướng 3 tháng
    "RON95_MA3",
    "Diesel_MA3",
    "Brent_MA3",
    "USDVND_MA3",

    # Quan hệ Brent - WTI
    "Brent_WTI_Spread_lag1",

    # Sự kiện
    "Dummy_Tet",
    "Dummy_Covid",

    # Mùa vụ
    "Month_sin",
    "Month_cos"
]

target_col = "CPI"


In [12]:
model_df = df[
    ["MonthYear", target_col] + feature_cols
].copy()

model_df = model_df[
    model_df["MonthYear"] >= pd.Period("2012-01", freq="M")
].reset_index(drop=True)

print("Kích thước:", model_df.shape)

print("\nSố giá trị thiếu:")
print(model_df.isna().sum())

model_df.head().round(2)

Kích thước: (156, 17)

Số giá trị thiếu:
MonthYear                0
CPI                      0
CPI_lag1                 0
CPI_lag2                 0
RON95_change_lag1        0
Diesel_change_lag1       0
Brent_change_lag1        0
USD_VND_change_lag1      0
RON95_MA3                0
Diesel_MA3               0
Brent_MA3                0
USDVND_MA3               0
Brent_WTI_Spread_lag1    0
Dummy_Tet                0
Dummy_Covid              0
Month_sin                0
Month_cos                0
dtype: int64


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_15012\1664041203.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  model_df.head().round(2)


,MonthYear,CPI,CPI_lag1,CPI_lag2,RON95_change_lag1,Diesel_change_lag1,Brent_change_lag1,USD_VND_change_lag1,RON95_MA3,Diesel_MA3,Brent_MA3,USDVND_MA3,Brent_WTI_Spread_lag1,Dummy_Tet,Dummy_Covid,Month_sin,Month_cos
0,2012-01,0.66,0.16,-0.01,0.00,0.00,-2.62,0.05,0.00,-1.11,-1.47,0.30,9.31,1,0,0.50,0.87
1,2012-02,0.23,0.66,0.16,0.00,0.00,2.61,0.07,0.00,0.00,0.37,0.18,10.42,0,0,0.87,0.50
2,2012-03,1.08,0.23,0.66,0.00,0.00,7.81,0.00,0.00,0.00,2.60,0.04,17.13,0,0,1.00,0.00
3,2012-04,2.67,1.08,0.23,5.92,3.95,5.13,0.00,1.97,1.32,5.18,0.02,19.29,0,0,0.87,-0.50
4,2012-05,1.32,2.67,1.08,1.34,0.91,-4.54,0.00,2.42,1.62,2.80,0.00,16.43,0,0,0.50,-0.87


## 7. Kiểm tra và lưu bộ dữ liệu đặc trưng

Bộ dữ liệu sau Feature Engineering được kiểm tra về số lượng quan sát, giá trị thiếu và trùng lặp thời gian trước khi lưu để sử dụng cho bước xây dựng mô hình.

In [13]:
print("Kích thước:", model_df.shape)
print("Số giá trị thiếu:", model_df.isna().sum().sum())
print("Số MonthYear trùng:", model_df["MonthYear"].duplicated().sum())

print(
    "Khoảng thời gian:",
    model_df["MonthYear"].min(),
    "→",
    model_df["MonthYear"].max()
)

Kích thước: (156, 17)
Số giá trị thiếu: 0
Số MonthYear trùng: 0
Khoảng thời gian: 2012-01 → 2024-12


In [14]:
output_df = model_df.copy()

output_df["MonthYear"] = output_df["MonthYear"].astype(str)

output_df.to_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/feature_dataset.csv",
    index=False
)